In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.prediction.clean_data_before_prediction import prepare_training_dataframe

In [ ]:
import pandas as pd
from nba_ou.create_training_data.create_df_to_predict import create_df_to_predict
# Create training data up to a specific date
date_to_train_1 = "2026-01-24"

safe_limit = "2024-10-11"  # Optional: specify start date


df_total_1 = create_df_to_predict(
    recent_limit_to_include=date_to_train_1, older_limit_to_include=safe_limit
)

date_to_train_2 = "2026-01-01"

df_total_2 = create_df_to_predict(
    recent_limit_to_include=date_to_train_2, older_limit_to_include=safe_limit
)


In [ ]:
import numpy as np
older_limit_to_include = "2025-12-16" # Optional: specify start date
max_limit_date = "2026-01-01" 

df_total_1['GAME_DATE'] = pd.to_datetime(df_total_1['GAME_DATE'])
df_eval_1 = df_total_1[(df_total_1['GAME_DATE'] >= pd.to_datetime(older_limit_to_include)) & (df_total_1['GAME_DATE'] <= pd.to_datetime(max_limit_date))]
df_eval_1['OVER_UNDER'] = np.where(df_eval_1['TOTAL_POINTS'] >= df_eval_1['TOTAL_OVER_UNDER_LINE'], 1, 0)
df_eval_1 = prepare_training_dataframe(df_eval_1)

In [ ]:

older_limit_to_include = "2025-12-16" # Optional: specify start date
max_limit_date = "2026-01-01" 

df_total_2['GAME_DATE'] = pd.to_datetime(df_total_2['GAME_DATE'])
df_eval_2 = df_total_2[(df_total_2['GAME_DATE'] >= pd.to_datetime(older_limit_to_include)) & (df_total_2['GAME_DATE'] <= pd.to_datetime(max_limit_date))]
df_eval_2['OVER_UNDER'] = np.where(df_eval_2['TOTAL_POINTS'] >= df_eval_2['TOTAL_OVER_UNDER_LINE'], 1, 0)
df_eval_2 = prepare_training_dataframe(df_eval_2)


# Predicitons Classifier

In [ ]:
from autogluon.tabular import TabularPredictor
predictor = TabularPredictor.load("/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/classifier_autoglue_3")

In [ ]:
df_predict = df_eval_1.drop(columns=['OVER_UNDER', 'TOTAL_POINTS'])
y_test = df_eval_1['OVER_UNDER']

In [ ]:
y_test_pred = predictor.predict(df_predict)
y_test_pred

In [ ]:
metrics = {
    "accuracy": accuracy_score(y_test, y_test_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_test_pred),
    "precision": precision_score(y_test, y_test_pred),
    "recall": recall_score(y_test, y_test_pred),
    "f1": f1_score(y_test, y_test_pred),
    "error_rate": 1 - accuracy_score(y_test, y_test_pred),
}

pd.Series(metrics).round(4)

In [ ]:
df_predict_2 = df_eval_2.drop(columns=['OVER_UNDER', 'TOTAL_POINTS'])
y_test_2 = df_eval_2['OVER_UNDER']

In [ ]:
y_test_pred_2 = predictor.predict(df_predict_2)
y_test_pred_2

In [ ]:
metrics_2 = {
    "accuracy": accuracy_score(y_test_2, y_test_pred_2),
    "balanced_accuracy": balanced_accuracy_score(y_test_2, y_test_pred_2),
    "precision": precision_score(y_test_2, y_test_pred_2),
    "recall": recall_score(y_test_2, y_test_pred_2),
    "f1": f1_score(y_test_2, y_test_pred_2),
    "error_rate": 1 - accuracy_score(y_test_2, y_test_pred_2),
}

pd.Series(metrics_2).round(4)